In [ ]:
import os
import rawpy
from PIL import Image
import piexif
from tqdm import tqdm
import numpy as np

def extract_gps_from_exif(exif_dict):
    """
    Extract GPS IFD and Image DateTime tag from EXIF dict.
    Returns a minimal EXIF dict with these tags only.
    """
    gps_ifd = exif_dict.get("GPS", {})

    zeroth_ifd = {}
    # Tag 0x0132 is DateTime in 0th IFD
    datetime_tag = 0x0132
    if datetime_tag in exif_dict.get("0th", {}):
        zeroth_ifd[datetime_tag] = exif_dict["0th"][datetime_tag]

    return {
        "0th": zeroth_ifd,
        "GPS": gps_ifd
    }
input_dir = "D:\\Rothaut_Thesis\\Mai\\raw"
output_dir = "D:\\Rothaut_Thesis\\Mai\\jpg"
os.makedirs(output_dir, exist_ok=True)

for filename in tqdm(sorted(os.listdir(input_dir))):
    if not filename.lower().endswith(".dng"):
        continue
    
    filename = filename.replace("._","")
    out_name = filename.replace(".dng", ".jpg").replace(".DNG", ".jpg")
    jpgs = os.listdir(output_dir)
    if out_name in jpgs:
        continue
    dng_path = input_dir +"/"+filename
    with rawpy.imread(dng_path) as raw:
        rgb = raw.postprocess(exp_shift=2, no_auto_bright=True)
    img = Image.fromarray(rgb)
    exif_dict = piexif.load(dng_path)
    gps_only_exif = extract_gps_from_exif(exif_dict)
    exif_bytes = piexif.dump(gps_only_exif)
    out_path = os.path.join(output_dir, out_name)
    img.save(out_path, "jpeg", exif=exif_bytes, quality=100)

print("GPS-only EXIF transfer complete.")

100%|██████████| 495/495 [09:58<00:00,  1.21s/it]

GPS-only EXIF transfer complete.


In [69]:
from PIL import Image
import numpy as np
def convert2to1(img1, img2, indir, outdir):
    numb = img1.split("-")[1]
    image1 = Image.open(os.path.join(indir, img1)).convert("L")
    image2 = Image.open(os.path.join(indir, img2)).convert("L")
    image1_array = np.array(image1)
    image2_array = np.array(image2)
    newg = 150
    image2_array[image2_array != 255] = 0
    image1_array[image1_array != 255] = 0
    image2_array[image2_array == 255] = newg
    merged = np.maximum(image1_array, image2_array)
    merged_im = Image.fromarray(merged, mode="L")
    merged_im.save(os.path.join(outdir,f"mask_{numb}.png"))

indir ="E:\Masterthesis_FIRO\Mai\masks"
outdir="C:/Users/dmz-user/Desktop/mask_gen/thesis/images/masks"
img1list = []
img2list = []
for filename in sorted(os.listdir(indir)):
    if filename.lower().endswith(".png"):
        if "-mit" in filename:
            img2list.append(filename)
        else:
            img1list.append(filename)
for i in range(len(img1list)):
    convert2to1(img1list[i], img2list[i], indir, outdir)

In [ ]:
import os
from PIL import Image

def get_task_and_tag(filename):
    # Extrahiere Task und Tag aus dem Dateinamen
    base_name = os.path.basename(filename)
    task, tag_with = base_name.split('-tag-')
    tag, number = tag_with.split('.')
    return task, tag, int(number)  # Gib den Task, Tag und die Zahl zurück

def merge_images(image1, image2):
    # Vereine zwei Bilder (z.B. nebeneinander)
    new_width = image1.width + image2.width
    new_height = max(image1.height, image2.height)
    new_image = Image.new('RGB', (new_width, new_height))
    
    # Füge beide Bilder nebeneinander hinzu
    new_image.paste(image1, (0, 0))
    new_image.paste(image2, (image1.width, 0))
    
    return new_image

def process_images(image_folder):
    # Erstelle ein Dictionary, um Bilder nach Task und Tag zu gruppieren
    image_dict = {}
    
    for filename in os.listdir(image_folder):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            task, tag, number = get_task_and_tag(filename)
            key = (task, tag)
            
            # Speichern der Bilder nach Task und Tag
            if key not in image_dict:
                image_dict[key] = []
            image_dict[key].append((number, filename))
    
    # Nun die Bilder mit der niedrigeren Nummer benennen und ggf. vereinen
    for (task, tag), images in image_dict.items():
        # Sortiere nach der Nummer
        images.sort(key=lambda x: x[0])
        
        # Das Bild mit der niedrigsten Nummer benennen
        first_image_filename = images[0][1]
        first_image = Image.open(os.path.join(image_folder, first_image_filename))
        
        if len(images) == 2:
            second_image_filename = images[1][1]
            second_image = Image.open(os.path.join(image_folder, second_image_filename))
            
            # Vereine die beiden Bilder
            merged_image = merge_images(first_image, second_image)
            merged_image.save(os.path.join(image_folder, f"{task}-tag-{tag}.jpg"))
        else:
            # Falls nur ein Bild vorhanden ist, belasse es
            first_image.save(os.path.join(image_folder, f"{task}-tag-{tag}.jpg"))

# Beispiel: den Pfad zum Ordner angeben
image_folder = "path/to/your/images"
process_images(image_folder)